In [8]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

def load_data(p='ml-100k/'):
    d = pd.read_csv(p + 'u.data', sep='\t', names=['u', 'i', 'r', 't'])
    info = pd.read_csv(p + 'u.user', sep='|', names=['u', 'age', 'g', 'occ', 'z'])
    u_list = sorted(d['u'].unique())
    u2i = {uid: i for i, uid in enumerate(u_list)}
    y = info.sort_values('u')['occ'].astype('category').cat.codes.values
    return d, u2i, y

def build_h(df, u2i):
    n = len(u2i)
    edges = []
    s = []
    for mid, g in df.groupby('i'):
        p_nodes = [u2i[uid] for uid in g[g['r'] >= 4]['u']]
        if len(p_nodes) >= 2:
            edges.append(p_nodes)
            s.append(1)
        n_nodes = [u2i[uid] for uid in g[g['r'] <= 2]['u']]
        if len(n_nodes) >= 2:
            edges.append(n_nodes)
            s.append(-1)
    h = np.zeros((n, len(edges)))
    for j, nodes in enumerate(edges):
        h[nodes, j] = 1
    return h, np.array(s)

def run_ashd(h, s_init, k=10, iters=50, dim=128):
    n, m = h.shape
    eps = 1e-10
    np.random.seed(42)
    x = np.random.normal(0, 0.1, (n, dim))
    x /= (np.linalg.norm(x, axis=1, keepdims=True) + eps)
    w = s_init.astype(float)
    de = np.sum(h, axis=0) + eps
    de_inv = np.diag(1.0 / de)
    
    for t in range(iters):
        dv = np.sum(h @ np.diag(np.abs(w)), axis=1) + eps
        dv_inv_sqrt = np.diag(1.0 / np.sqrt(dv))
        p = dv_inv_sqrt @ h @ np.diag(w) @ de_inv @ h.T @ dv_inv_sqrt
        x = p @ x
        x /= (np.linalg.norm(x, axis=1, keepdims=True) + eps)
        
        if t > 0 and t % 2 == 0:
            for j in range(m):
                idx = np.where(h[:, j] > 0)[0]
                if len(idx) >= 2:
                    sim = cosine_similarity(x[idx])
                    avg = (np.sum(sim) - len(idx)) / (len(idx) * (len(idx) - 1) + eps)
                    w[j] = np.clip(w[j] + 0.1 * avg, -1, 1)
    return x

def get_mod(h, labs):
    n, m = h.shape
    dv = np.sum(h, axis=1)
    de = np.sum(h, axis=0)
    de[de == 0] = 1
    q = 0.0
    for c in np.unique(labs):
        nodes = np.where(labs == c)[0]
        vol = np.sum(dv[nodes])
        for e in range(m):
            overlap = len(np.intersect1d(np.where(h[:, e] > 0)[0], nodes))
            if overlap > 0:
                q += (overlap / de[e]) - (vol / (2 * m))
    return q / m

if __name__ == "__main__":
    df, u_map, y_true = load_data()
    h, s = build_h(df, u_map)
    
    emb = run_ashd(h, s)
    km = KMeans(n_clusters=10, n_init=20, random_state=42)
    y_pred = km.fit_predict(emb)
    
    print("NMI:", normalized_mutual_info_score(y_true, y_pred))
    print("Modularity:", get_mod(h, y_pred))
    print("Silhouette:", silhouette_score(emb, y_pred))

NMI: 0.06286035713734889
Modularity: -8.334844088631
Silhouette: 0.5000948745140784
